In [12]:
# CUSTOM CNN ARCHITECTURE

In [13]:
import torch
import torch.nn as nn

In [14]:


class PlantDiseaseCNN(nn.Module):

    def __init__(self):
        super().__init__()

model = PlantDiseaseCNN()

print(model)

PlantDiseaseCNN()


In [15]:
import torch
import torch.nn as nn

conv = nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3
)

print(conv)

print(conv.weight.shape)

Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
torch.Size([32, 3, 3, 3])


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : NVIDIA GeForce RTX 4050 Laptop GPU


In [17]:
class PlantDiseaseCNN(nn.Module):

    def __init__(self):

        super().__init__()

        # Block 1
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.relu1 = nn.ReLU()

        self.pool1 = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # Block 2
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.relu2 = nn.ReLU()

        self.pool2 = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # Block 3
        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            padding=1
        )

        self.relu3 = nn.ReLU()

        self.pool3 = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # Flatten

        self.flatten = nn.Flatten()

        # Fully Connected Layers

        self.fc1 = nn.Linear(
            128 * 28 * 28,
            512
        )

        self.relu4 = nn.ReLU()

        self.dropout = nn.Dropout(0.5)

        self.fc2 = nn.Linear(
            512,
            38
        )

    def forward(self, x):

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)

        x = self.flatten(x)

        x = self.fc1(x)

        x = self.relu4(x)

        x = self.dropout(x)

        x = self.fc2(x)

        return x

In [18]:
model = PlantDiseaseCNN().to(device)

dummy = torch.randn(1,3,224,224).to(device)

output = model(dummy)

print(output.shape)

torch.Size([1, 38])


In [19]:
x = torch.randn(1, 3, 224, 224).to(device)

print("Input :", x.shape)

x = model.conv1(x)
print("Conv1 :", x.shape)

x = model.relu1(x)

x = model.pool1(x)
print("Pool1 :", x.shape)

x = model.conv2(x)
print("Conv2 :", x.shape)

x = model.relu2(x)

x = model.pool2(x)
print("Pool2 :", x.shape)

x = model.conv3(x)
print("Conv3 :", x.shape)

x = model.relu3(x)

x = model.pool3(x)
print("Pool3 :", x.shape)

x = model.flatten(x)
print("Flatten :", x.shape)

Input : torch.Size([1, 3, 224, 224])
Conv1 : torch.Size([1, 32, 224, 224])
Pool1 : torch.Size([1, 32, 112, 112])
Conv2 : torch.Size([1, 64, 112, 112])
Pool2 : torch.Size([1, 64, 56, 56])
Conv3 : torch.Size([1, 128, 56, 56])
Pool3 : torch.Size([1, 128, 28, 28])
Flatten : torch.Size([1, 100352])


In [20]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total Parameters:", total_params)
print("Trainable Parameters:", trainable_params)

Total Parameters: 51493478
Trainable Parameters: 51493478


In [21]:
for name, parameter in model.named_parameters():
    print(name, parameter.shape, parameter.numel())

conv1.weight torch.Size([32, 3, 3, 3]) 864
conv1.bias torch.Size([32]) 32
conv2.weight torch.Size([64, 32, 3, 3]) 18432
conv2.bias torch.Size([64]) 64
conv3.weight torch.Size([128, 64, 3, 3]) 73728
conv3.bias torch.Size([128]) 128
fc1.weight torch.Size([512, 100352]) 51380224
fc1.bias torch.Size([512]) 512
fc2.weight torch.Size([38, 512]) 19456
fc2.bias torch.Size([38]) 38


In [25]:
criterion = nn.CrossEntropyLoss()

# Dummy batch
images = torch.randn(32, 3, 224, 224).to(device)

# Random class labels: 0 to 37
labels = torch.randint(0, 38, (32,)).to(device)

# Forward pass
output = model(images)

# Loss
loss = criterion(output, labels)

# Predictions
predictions = output.argmax(dim=1)

# Accuracy
accuracy = (predictions == labels).float().mean()

print("Output shape :", output.shape)
print("Labels shape :", labels.shape)
print("Loss         :", loss.item())
print("Accuracy     :", accuracy.item())

Output shape : torch.Size([32, 38])
Labels shape : torch.Size([32])
Loss         : 3.6294236183166504
Accuracy     : 0.03125


In [26]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Dummy batch
images = torch.randn(
    32, 3, 224, 224
).to(device)

labels = torch.randint(
    0, 38, (32,)
).to(device)


# =========================
# TRAINING STEP
# =========================

# 1. Reset old gradients
optimizer.zero_grad()

# 2. Forward pass
output = model(images)

# 3. Calculate loss
loss = criterion(output, labels)

# 4. Backpropagation
loss.backward()

# 5. Update weights
optimizer.step()


print("Loss:", loss.item())

Loss: 3.615938663482666


In [27]:
print(model.conv1.weight.grad)

tensor([[[[ 4.9593e-04,  1.5101e-03,  4.0709e-03],
          [-6.8504e-04,  1.6374e-03, -6.9317e-04],
          [ 8.2863e-04,  1.9468e-03, -2.8311e-03]],

         [[ 4.6845e-04, -5.2927e-04, -1.2980e-03],
          [-2.0228e-03,  2.0143e-03,  3.2150e-03],
          [ 1.1575e-03, -3.6891e-03, -1.1555e-03]],

         [[-4.7798e-04, -2.1689e-03,  2.5749e-03],
          [-1.9524e-04,  1.8746e-03,  2.4145e-03],
          [ 4.3913e-04, -2.8107e-03, -3.7169e-03]]],


        [[[-1.2989e-03, -1.8376e-03,  1.4679e-03],
          [-3.8844e-05,  1.9409e-03, -4.2005e-03],
          [-2.2419e-03, -3.4036e-03, -5.1397e-03]],

         [[-1.0688e-03, -3.0388e-03,  2.2599e-03],
          [-6.0678e-04,  9.9595e-04, -2.7645e-03],
          [-2.1378e-03,  3.6711e-04,  5.7755e-04]],

         [[-8.2145e-05, -4.0697e-03,  2.0681e-03],
          [ 1.0855e-03, -5.1749e-03,  3.7534e-04],
          [ 6.6486e-04, -6.1214e-04,  7.8665e-04]]],


        [[[-5.2505e-04,  7.8187e-04,  4.2252e-03],
          [ 1.8

In [28]:
print(model.conv1.weight.shape)

torch.Size([32, 3, 3, 3])
